In [ ]:
import os
import pickle
import numpy as np
import pandas as po
import matplotlib.pyplot as plt

In [ ]:
from sklearn.metrics import confusion_matrix, roc_curve, roc_auc_score, precision_recall_curve, auc

In [ ]:
df_cm = po.DataFrame()
train_hist = {}
base_dir = 'results/final_plots/'
for model in os.listdir(base_dir):
    train_hist[model] = {}

    with open(base_dir+model + '/plots/predictions.npz', 'rb') as f:
        data = np.load(f)
    
        y_true = data['y_true']
        y_pred = data['y_pred']

    with open(base_dir+model + '/plots/history.pkl', 'rb') as f:
        train_hist[model]['history'] = pickle.load(f)

    train_hist[model]['y_true'] = np.argmax(y_true, axis = 1)
    train_hist[model]['y_pred'] = np.argmax(y_pred, axis = 1)
    train_hist[model]['y_prob_true'] = y_pred[:, 1]

    tn, fp, fn, tp = confusion_matrix(train_hist[model]['y_true'], train_hist[model]['y_pred']).ravel()
    precision, recall, thresholds = precision_recall_curve(y_true.argmax(axis=1), y_pred.argmax(axis=1))
    row = {'model': model, 'TN': tn, 'FP': fp, 'FN': fn, 'TP': tp, 'precision': precision, 'recall': recall}
    df_cm = df_cm.append(row, ignore_index=True)

#df_cm.to_csv('results/confusion_matrix.csv', index=False)

In [ ]:
colors = ['orange', 'blue', 'green', 'red', 'grey', 'brown']

In [ ]:
train_hist.keys()

In [ ]:
del train_hist['transformer']

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC

In [ ]:
df = po.read_csv('/home/varun/Desktop/Projects/EnergyTheft/data/processed/EnergyConsumption_processed.csv')
X = df.drop(['FLAG'], axis=1)
y = df['FLAG']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
clf_rf = RandomForestClassifier(random_state=42, n_jobs=-1)
clf_rf.fit(X_train, y_train)
y_pred_rf = clf_rf.predict(X_test)
fpr_rf, tpr_rf, thresholds_rf = roc_curve(y_test, y_pred_rf)
precision_rf, recall_rf, thresholds_rf = precision_recall_curve(y_test, y_pred_rf)
auc_rf = auc(fpr_rf, tpr_rf)

In [ ]:
clf_svc = SVC(random_state=42)
clf_svc.fit(X_train, y_train)
y_pred_svc = clf_svc.predict(X_test)
fpr_svc, tpr_svc, thresholds_svc = roc_curve(y_test, y_pred_svc)
precision_svc, recall_svc, thresholds_svc = precision_recall_curve(y_test, y_pred_svc)
auc_svc = auc(fpr_svc, tpr_svc)

In [ ]:
clf_gbm = SVC(random_state=42)
clf_gbm.fit(X_train, y_train)
y_pred_gbm = clf_gbm.predict(X_test)
fpr_gbm, tpr_gbm, thresholds_gbm = roc_curve(y_test, y_pred_gbm)
precision_gbm, recall_gbm, thresholds_gbm = precision_recall_curve(y_test, y_pred_gbm)
auc_gbm = auc(fpr_gbm, tpr_gbm)

In [ ]:
plt.figure(figsize=(10, 5))
for i, model in enumerate(train_hist):
    lr_fpr, lr_tpr, _ = roc_curve(train_hist[model]['y_true'], train_hist[model]['y_prob_true'])
    auc = roc_auc_score(train_hist[model]['y_true'], train_hist[model]['y_prob_true'])
    plt.plot(lr_fpr, lr_tpr, color=colors[i], marker='.', label=model+' AUC = {:.2f}'.format(auc))

plt.plot(fpr_rf, tpr_rf, color=colors[4], marker='.', label="RandomForests"+' AUC = {:.2f}'.format(auc_rf))
#plt.plot(fpr_gbm, tpr_gbm, color=colors[5], marker='.', label=model+' AUC = {:.2f}'.format(auc_gbm))

plt.legend(fontsize=10)
plt.xlabel('False Positive Rate', fontsize=20)
plt.ylabel('True Positive Rate', fontsize=20)
plt.savefig('results/final_roc_curve.png')
plt.close()

In [ ]:
plt.figure(figsize=(10, 5))
for i, model in enumerate(train_hist):
    plt.plot(train_hist[model]['history']['recall'], train_hist[model]['history']['precision'], color=colors[i], marker='.', label=model)

plt.plot(precision_rf, recall_rf, color=colors[4], marker='.', label="RandomForests")
#plt.plot(precision_gbm, recall_gbm, color=colors[5], marker='.', label=model+' AUC = {:.2f}'.format(auc_gbm))

plt.legend(fontsize=10)
plt.xlabel('False Positive Rate', fontsize=20)
plt.ylabel('True Positive Rate', fontsize=20)
plt.savefig('results/final_prec_rec_curve.png')
plt.close()

In [ ]:
metric = 'precision'
plt.figure(figsize=(20, 10))
for i, model in enumerate(train_hist):
    plt.plot(train_hist[model]['history'][metric][:15], color=colors[i], marker='.', label=model)
plt.xlabel('Epochs', fontsize=20)
plt.ylabel('Precision', fontsize=20)
plt.legend(fontsize=20)
#plt.show()
plt.savefig('results/{}.png'.format(metric))
plt.close()

In [ ]:
metric = 'val_precision'
plt.figure(figsize=(20, 10))
for i, model in enumerate(train_hist):
    plt.plot(train_hist[model]['history'][metric][:15], color=colors[i], marker='.', label=model)
plt.xlabel('Epochs', fontsize=20)
plt.ylabel('Precision', fontsize=20)
plt.legend(fontsize=20)
#plt.show()
plt.savefig('results/{}.png'.format(metric))
plt.close()

In [ ]:
metric = 'recall'
plt.figure(figsize=(20, 10))
for i, model in enumerate(train_hist):
    plt.plot(train_hist[model]['history'][metric][:15], color=colors[i], marker='.', label=model)
plt.xlabel('Epochs', fontsize=20)
plt.ylabel('Recall', fontsize=20)
plt.legend(fontsize=20)
#plt.show()
plt.savefig('results/{}.png'.format(metric))
plt.close()

In [ ]:
metric = 'val_recall'
plt.figure(figsize=(20, 10))
for i, model in enumerate(train_hist):
    plt.plot(train_hist[model]['history'][metric][:15], color=colors[i], marker='.', label=model)
plt.xlabel('Epochs', fontsize=20)
plt.ylabel('Recall', fontsize=20)
plt.legend(fontsize=20)
#plt.show()
plt.savefig('results/{}.png'.format(metric))
plt.close()

In [ ]:
plt.figure(figsize=(20, 10))
for i, model in enumerate(train_hist):
    plt.plot(train_hist[model]['history']['loss'][:15], color=colors[i], marker='.', label=model)
plt.xlabel('Epochs', fontsize=20)
plt.ylabel('Training Loss', fontsize=20)
plt.legend(fontsize=20)
plt.savefig('results/train_loss.png')
plt.close()

In [ ]:
plt.figure(figsize=(20, 10))
for i, model in enumerate(train_hist):
    plt.plot(train_hist[model]['history']['val_loss'][:15], color=colors[i], marker='.', label=model)
plt.xlabel('Epochs', fontsize=20)
plt.ylabel('Val Loss', fontsize=20)
plt.legend(fontsize=20)
plt.savefig('results/val_loss.png')
plt.close()

In [ ]:
plt.figure(figsize=(20, 10))
for i, model in enumerate(train_hist):
    lr_fpr, lr_tpr, _ = roc_curve(train_hist[model]['y_true'], train_hist[model]['y_prob_true'])
    auc = roc_auc_score(train_hist[model]['y_true'], train_hist[model]['y_prob_true'])
    plt.plot(lr_fpr, lr_tpr, color=colors[i], marker='.', label=model+' AUC = {:.2f}'.format(auc))

plt.legend(fontsize=20)
plt.xlabel('False Positive Rate', fontsize=20)
plt.ylabel('True Positive Rate', fontsize=20)
plt.savefig('results/roc_curve.png')
plt.close()

In [ ]:
_, counts = np.unique(np.argmax(y_val, axis = 1), return_counts=True)
counts

In [ ]:
total_pos_val = counts[0]
total_neg_val = counts[1]

In [ ]:
total_pos_train

In [ ]:
train_hist[model]['history']['true_positives']

In [ ]:
plt.figure(figsize=(20, 10))
for i, model in enumerate(train_hist):
    plt.plot(train_hist[model]['history']['binary_accuracy'][:15], color=colors[i], marker='.', label=model)
plt.xlabel('Epochs', fontsize=20)
plt.ylabel('Accuracy', fontsize=20)
plt.legend(fontsize=20)
plt.savefig('results/accuracy.png')
plt.close()

In [ ]:
train_hist['lstmfcn']['history'].keys()

In [ ]:
metric = 'auc'
plt.figure(figsize=(20, 10))
for i, model in enumerate(train_hist):
    plt.plot(train_hist[model]['history'][metric][:15], color=colors[i], marker='.', label=model)
plt.xlabel('Epochs', fontsize=20)
plt.ylabel('Area Under the Curve', fontsize=20)
plt.legend(fontsize=20)
#plt.show()
plt.savefig('results/{}.png'.format(metric))
plt.close()

In [ ]:
metric = 'val_auc'
plt.figure(figsize=(20, 10))
for i, model in enumerate(train_hist):
    plt.plot(train_hist[model]['history'][metric][:15], color=colors[i], marker='.', label=model)
plt.xlabel('Epochs', fontsize=20)
plt.ylabel('Area Under the Curve', fontsize=20)
plt.legend(fontsize=20)
#plt.show()
plt.savefig('results/{}.png'.format(metric))
plt.close()

In [ ]:
metric = 'val_matthews_correlation'
plt.figure(figsize=(20, 10))
for i, model in enumerate(train_hist):
    plt.plot(train_hist[model]['history'][metric][:15], color=colors[i], marker='.', label=model)
plt.xlabel('Epochs', fontsize=20)
plt.ylabel('MCC', fontsize=20)
plt.legend(fontsize=20)
#plt.show()
plt.savefig('results/{}.png'.format(metric))
plt.close()

In [ ]:
metric = 'matthews_correlation'
plt.figure(figsize=(20, 10))
for i, model in enumerate(train_hist):
    plt.plot(train_hist[model]['history'][metric][:15], color=colors[i], marker='.', label=model)
plt.xlabel('Epochs', fontsize=20)
plt.ylabel('MCC', fontsize=20)
plt.legend(fontsize=20)
#plt.show()
plt.savefig('results/{}.png'.format(metric))
plt.close()

In [ ]:
metric = 'matthews_correlation'
plt.figure(figsize=(20, 10))
for i, model in enumerate(train_hist):
    plt.plot(train_hist[model]['history'][metric][:15], color=colors[i], marker='.', label=model)
plt.xlabel('Epochs', fontsize=20)
plt.ylabel('MCC', fontsize=20)
plt.legend(fontsize=20)
#plt.show()
plt.savefig('results/{}.png'.format(metric))
plt.close()

In [ ]:
from preprocessing import get_standard_data

In [ ]:
(X_train, y_train), (X_val, y_val), (X_test, y_test) = get_standard_data(0.6, 0.2, 0.2)

In [ ]:
_, counts = np.unique(np.argmax(y_train, axis = 1), return_counts=True)
counts

In [ ]:
total_pos_train = counts[0]
total_neg_train = counts[1]

In [ ]:
metric = 'true_positives'
plt.figure(figsize=(20, 10))
for i, model in enumerate(train_hist):
    plt.plot(train_hist[model]['history'][metric][:15]/total_pos_train, color=colors[i], marker='.', label=model)
plt.xlabel('Epochs', fontsize=20)
plt.ylabel('TPR', fontsize=20)
plt.legend(fontsize=20)
#plt.show()
#plt.savefig('results/{}.png'.format(metric))
#plt.close()

In [ ]:
metric = 'val_true_positives'
plt.figure(figsize=(20, 10))
for i, model in enumerate(train_hist):
    plt.plot(train_hist[model]['history'][metric][:15]/true_pos_val, color=colors[i], marker='.', label=model)
plt.xlabel('Epochs', fontsize=20)
plt.ylabel('TPR', fontsize=20)
plt.legend(fontsize=20)
#plt.show()
plt.savefig('results/{}.png'.format(metric))
plt.close()

In [ ]:
metric = 'false_positives'
plt.figure(figsize=(20, 10))
for i, model in enumerate(train_hist):
    print(train_hist[model]['history'][metric][:15])
    plt.plot(train_hist[model]['history'][metric][:15]/true_neg_train, color=colors[i], marker='.', label=model)
plt.xlabel('Epochs', fontsize=20)
plt.ylabel('FPR', fontsize=20)
plt.legend(fontsize=20)
#plt.show()
#plt.savefig('results/{}.png'.format(metric))
#plt.close()

In [ ]:
metric = 'val_true_negatives'
plt.figure(figsize=(20, 10))
for i, model in enumerate(train_hist):
    _, counts = np.unique(train_hist[model]['y_true'], return_counts=True)
    total_pos = counts[0]
    plt.plot(train_hist[model]['history'][metric][:15]/total_pos, color=colors[i], marker='.', label=model)
plt.xlabel('Epochs', fontsize=20)
plt.ylabel('FPR', fontsize=20)
plt.legend(fontsize=20)
plt.show()
#plt.savefig('results/{}.png'.format(metric))
#plt.close()